In [1]:
import numpy as np
import pandas as pd
import os
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.model_selection import ParameterGrid
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

import config as cnf 

In [2]:
df_train                = pd.read_csv(cnf.path['train_path'])
df_test                 = pd.read_csv(cnf.path['test_path'])
df_gender_submission    = pd.read_csv(cnf.path['gender_submission_path'])

### Логистическая регрессия

In [3]:
def predobr(df_train, df_test, df_gender_submission):
    df_train_priz = df_train[['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']].copy()
    df_train_priz['Age'] = df_train_priz['Age'].fillna(df_train_priz['Age'].median())
    df_train_priz['Embarked'] = df_train_priz['Embarked'].fillna('S')
    df_train_priz = pd.get_dummies(df_train_priz, dtype=float, drop_first=True)
    
    X = df_train_priz.drop('Survived', axis=1)
    y = df_train_priz['Survived']
    return X, y


def Split(model, X, y, train_index, test_index, scaler=False):
    current_X_train = X.iloc[train_index]
    current_y_train = y.iloc[train_index]
    current_X_test  = X.iloc[test_index]
    current_y_test  = y.iloc[test_index]
    
    if scaler:
        scaler  = StandardScaler()
        scaler.fit(current_X_train)
        current_X_train = scaler.transform(current_X_train)
        current_X_test = scaler.transform(current_X_test)
    
    model.fit(current_X_train, current_y_train)
    train_accuracy = model.score(current_X_train, current_y_train, sample_weight=None)
    test_accuracy = model.score(current_X_test, current_y_test, sample_weight=None)
    
    array_train_accuracy.append(train_accuracy)
    array_test_accuracy.append(test_accuracy)
    
    return array_train_accuracy, array_test_accuracy


def mean_accuracy(array_train_accuracy, array_test_accuracy):
        mean_train_accuracy = float(np.mean(array_train_accuracy))
        mean_train_std = np.std(array_train_accuracy)

        mean_test_accuracy = float(np.mean(array_test_accuracy))
        mean_test_std = np.std(array_test_accuracy)
        print("Train_accuracy: ", mean_train_accuracy, "Test_accuracy: ", mean_test_accuracy)
        
        return mean_train_accuracy, mean_train_std, mean_test_accuracy, mean_test_std
    
    
def parametrs(params):
    for key, value in params.items():   
        if type(value) != list:
            params[key] = [value]

    return list(ParameterGrid(params))


In [7]:
X, y = predobr(df_train, df_test, df_gender_submission)

combo_param = parametrs(cnf.RandomForestParam) + parametrs(cnf.LogRegParam)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = []

for combo in combo_param:
    name_model = combo['name_model']
    
    if name_model == 'RandomForest':
        n_estimators = combo['n_estimators']
        max_depth = combo['max_depth']
        min_samples_leaf = combo['min_samples_leaf']

        print("Количество деревьев: ", n_estimators, " | Максимальная глубина: ", max_depth, " | Минимум на листе :", min_samples_leaf)

        array_train_accuracy = []
        array_test_accuracy = []

        for i, (train_index, test_index) in enumerate(skf.split(X, y)):

            model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42, min_samples_leaf=min_samples_leaf)
            array_train_accuracy, array_test_accuracy = Split(model, X, y, train_index, test_index, scaler=False)

        mean_train_accuracy, mean_train_std, mean_test_accuracy, mean_test_std = mean_accuracy(array_train_accuracy, array_test_accuracy)

        results.append({
            'name_model'            :   name_model
            , 'n_estimator'         :   n_estimators
            , 'max_depth'           :   max_depth
            , 'min_samples_leaf'    :   min_samples_leaf
            , 'train_accuracy'      :   mean_train_accuracy
            , 'train_std'           :   mean_train_std
            , 'test_accuracy'       :   mean_test_accuracy
            , 'test_std'            :   mean_test_std
        })
    
    elif name_model == 'LogReg':
        
        print("LogReg")
        
        for i, (train_index, test_index) in enumerate(skf.split(X, y)):

            model = LogisticRegression(random_state=42)
            array_train_accuracy, array_test_accuracy = Split(model, X, y, train_index, test_index, scaler=True)
            
            
        mean_train_accuracy, mean_train_std, mean_test_accuracy, mean_test_std = mean_accuracy(array_train_accuracy, array_test_accuracy)

        results.append({
            'name_model'            :   name_model
            , 'n_estimator'         :   0
            , 'max_depth'           :   0
            , 'min_samples_leaf'    :   0
            , 'train_accuracy'      :   mean_train_accuracy
            , 'train_std'           :   mean_train_std
            , 'test_accuracy'       :   mean_test_accuracy
            , 'test_std'            :   mean_test_std
        })
    

Количество деревьев:  100  | Максимальная глубина:  8  | Минимум на листе : 1
Train_accuracy:  0.9158229194572701 Test_accuracy:  0.8361245370660976
Количество деревьев:  100  | Максимальная глубина:  8  | Минимум на листе : 2
Train_accuracy:  0.8947795357486171 Test_accuracy:  0.8372355784319879
Количество деревьев:  100  | Максимальная глубина:  8  | Минимум на листе : 3
Train_accuracy:  0.8835581574924752 Test_accuracy:  0.8349946644906158
Количество деревьев:  100  | Максимальная глубина:  9  | Минимум на листе : 1
Train_accuracy:  0.9326587295333848 Test_accuracy:  0.8428535559600778
Количество деревьев:  100  | Максимальная глубина:  9  | Минимум на листе : 2
Train_accuracy:  0.9051617630836629 Test_accuracy:  0.8327537505492437
Количество деревьев:  100  | Максимальная глубина:  9  | Минимум на листе : 3
Train_accuracy:  0.8880458420662812 Test_accuracy:  0.8394827694432239
Количество деревьев:  100  | Максимальная глубина:  10  | Минимум на листе : 1
Train_accuracy:  0.94219983

### Случайный лес

In [ ]:
df_train_priz = df_train[['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']].copy()
df_train_priz['Age'] = df_train_priz['Age'].fillna(df_train_priz['Age'].median())
df_train_priz['Embarked'] = df_train_priz['Embarked'].fillna('S')
df_train_priz = pd.get_dummies(df_train_priz, dtype=float, drop_first=True)

X = df_train_priz.drop('Survived', axis=1)
y = df_train_priz['Survived']

combo_param = list(ParameterGrid(cnf.param))
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = []

for combo in combo_param:
    n_estimators = combo['n_estimators']
    max_depth = combo['max_depth']
    min_samples_leaf = combo['min_samples_leaf']
    
    print("Количество деревьев: ", n_estimators, " | Максимальная глубина: ", max_depth, " | Минимум на листе :", min_samples_leaf)
    
    array_train_accuracy = []
    array_test_accuracy = []
    
    for i, (train_index, test_index) in enumerate(skf.split(X, y)):
        current_X_train, current_y_train, current_X_test, current_y_test = Split(X, y, train_index, test_index)

        model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42, min_samples_leaf=min_samples_leaf)

        model.fit(current_X_train, current_y_train)
        train_accuracy = model.score(current_X_train, current_y_train, sample_weight=None)
        test_accuracy = model.score(current_X_test, current_y_test, sample_weight=None)
        
        array_train_accuracy.append(train_accuracy)
        array_test_accuracy.append(test_accuracy)

    mean_train_accuracy = float(np.mean(array_train_accuracy))
    mean_train_std = np.std(array_train_accuracy)
    
    mean_test_accuracy = float(np.mean(array_test_accuracy))
    mean_test_std = np.std(array_test_accuracy)
    print("Train_accuracy: ", mean_train_accuracy, "Test_accuracy: ", mean_test_accuracy)
    
    results.append({
        'name_model'            :  'RandomForest'
        , 'n_estimator'         :   n_estimators
        , 'max_depth'           :   max_depth
        , 'min_samples_leaf'    :   min_samples_leaf
        , 'train_accuracy'      :   mean_train_accuracy
        , 'train_std'           :   mean_train_std
        , 'test_accuracy'       :   mean_test_accuracy
        , 'test_std'            :   mean_test_std
    })

Количество деревьев:  100  | Максимальная глубина:  5  | Минимум на листе : 1
Train_accuracy:  0.8496080022692531 Test_accuracy:  0.8338710689849979
Количество деревьев:  100  | Максимальная глубина:  5  | Минимум на листе : 2
Train_accuracy:  0.8490458105488756 Test_accuracy:  0.8293704098926622
Количество деревьев:  100  | Максимальная глубина:  5  | Минимум на листе : 3
Train_accuracy:  0.8473612052255858 Test_accuracy:  0.8249011361496453
Количество деревьев:  100  | Максимальная глубина:  6  | Минимум на листе : 1
Train_accuracy:  0.8720550924247916 Test_accuracy:  0.8349946644906158
Количество деревьев:  100  | Максимальная глубина:  6  | Минимум на листе : 2
Train_accuracy:  0.8630749956663568 Test_accuracy:  0.8327537505492437
Количество деревьев:  100  | Максимальная глубина:  6  | Минимум на листе : 3
Train_accuracy:  0.8571828167105284 Test_accuracy:  0.8338710689849977
Количество деревьев:  100  | Максимальная глубина:  7  | Минимум на листе : 1
Train_accuracy:  0.893939596

In [5]:
results = pd.DataFrame(results)

In [6]:
results['differenc'] = results['train_accuracy'] - results['test_accuracy']
results.sort_values('test_accuracy', ascending=False).head(50)

,name_model,n_estimator,max_depth,min_samples_leaf,train_accuracy,train_std,test_accuracy,test_std,differenc
3,RandomForest,100,9,1,0.932659,0.004454,0.842854,0.012525,0.089805
8,RandomForest,100,10,3,0.891694,0.003504,0.840613,0.014193,0.051081
6,RandomForest,100,10,1,0.942200,0.001371,0.840606,0.012351,0.101593
5,RandomForest,100,9,3,0.888046,0.004415,0.839483,0.019818,0.048563
7,RandomForest,100,10,2,0.909371,0.002893,0.837248,0.016013,0.072123
1,RandomForest,100,8,2,0.894780,0.004647,0.837236,0.022066,0.057544
0,RandomForest,100,8,1,0.915823,0.006486,0.836125,0.014097,0.079698
2,RandomForest,100,8,3,0.883558,0.005809,0.834995,0.020443,0.048563
4,RandomForest,100,9,2,0.905162,0.002320,0.832754,0.018059,0.072408
9,LogReg,0,0,0,0.847643,0.044185,0.817064,0.029004,0.030579
